# 📘 Notebook 04 — GRPO Reinforcement Learning Fine-tuning

**Goals:**
1. Load SFT-trained model (LoRA adapter)
2. Define rule-based reward functions
3. Train with GRPO (same method DeepSeek-R1 used)
4. Save final model

**Runtime:** ~2 hrs on Kaggle T4 | **GPU:** Required (16GB)

## 1. Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import (
    SFT_LORA_SAVE_DIR, GRPO_OUTPUT_DIR, GRPO_FINAL_SAVE_DIR,
    GRPO_LEARNING_RATE, GRPO_BATCH_SIZE, GRPO_GRAD_ACCUM_STEPS,
    GRPO_NUM_GENERATIONS, GRPO_MAX_NEW_TOKENS, GRPO_EPOCHS,
    COT_DATA_PATH,
)

print(f"SFT model: {SFT_LORA_SAVE_DIR}")
print(f"GRPO output: {GRPO_OUTPUT_DIR}")
print(f"Final save: {GRPO_FINAL_SAVE_DIR}")

## 2. Load SFT-Trained Model

In [ ]:
from src.model_utils import load_finetuned_model

model, tokenizer = load_finetuned_model(SFT_LORA_SAVE_DIR)
print("✅ SFT model loaded for GRPO training")

## 3. Load Training Data

In [ ]:
from src.data_utils import load_cot_dataset

cot_dataset = load_cot_dataset(COT_DATA_PATH)
print(f"Training samples: {len(cot_dataset)}")

## 4. Inspect Reward Functions

In [ ]:
from src.reward_functions import (
    reward_has_legal_citation,
    reward_has_reasoning,
    reward_bilingual,
    ALL_REWARD_FUNCS,
)

# Test with sample completions
test_completions = [
    "Under Section 302 IPC, the accused is guilty. <think>Therefore, applying the ratio decidendi...</think> अतः दोषी है।",
    "The person did something bad.",
]

print("=== Reward Function Tests ===")
print(f"Legal citation: {reward_has_legal_citation(test_completions)}")
print(f"Reasoning:      {reward_has_reasoning(test_completions)}")
print(f"Bilingual:      {reward_bilingual(test_completions)}")

## 5. Configure & Run GRPO Training

> **⏱️ This takes ~2 hours on a Kaggle T4 GPU.**
>
> GRPO generates multiple completions per prompt, rewards the best ones, 
> and updates the model — much lighter than PPO.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    learning_rate=GRPO_LEARNING_RATE,
    per_device_train_batch_size=GRPO_BATCH_SIZE,
    gradient_accumulation_steps=GRPO_GRAD_ACCUM_STEPS,
    num_generations=GRPO_NUM_GENERATIONS,
    max_new_tokens=GRPO_MAX_NEW_TOKENS,
    num_train_epochs=GRPO_EPOCHS,
    fp16=True,
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=ALL_REWARD_FUNCS,
    args=grpo_config,
    train_dataset=cot_dataset,
)

print("🚀 Starting GRPO training...")
grpo_trainer.train()

## 6. Save Final Model

In [ ]:
grpo_trainer.save_model(GRPO_FINAL_SAVE_DIR)
print(f"✅ GRPO training complete. Final model saved to: {GRPO_FINAL_SAVE_DIR}")

## 7. Quick Validation

In [ ]:
from src.model_utils import generate_judgment

sample_facts = """The accused was charged under Section 302 IPC for
allegedly murdering his neighbor during a property dispute.
The prosecution presented three eyewitnesses and forensic evidence.
The defense argued it was a case of self-defense under Section 96 IPC."""

print("=== GRPO-TUNED MODEL OUTPUT ===")
output = generate_judgment(model, tokenizer, sample_facts)
print(output)

## 8. Summary

✅ **Completed:**
- SFT model loaded with LoRA adapter
- GRPO training with 3 reward functions (citation, reasoning, bilingual)
- Final model saved

**Next → Notebook 05:** Full evaluation suite (ROUGE, BERTScore, LLM Judge)